# 01 — Naive RAG

```
╔═══════════════════════════════════════════════════════════╗
║                    1. NAIVE RAG                            ║
║                                                            ║
║  Documents ──► Chunks ──► Embedding Model ──► Vector DB   ║
║                                  ▲                │        ║
║  Query ────────────────────────► │           top-k│        ║
║                                               │        ║
║  Response ◄── Generative Model ◄── Prompt Template ◄──┘   ║
╚═══════════════════════════════════════════════════════════╝
```

## What is Naive RAG?

**Naive RAG** (also called basic RAG or vanilla RAG) is the foundational pattern:

1. **Index** — Split documents into chunks, embed each chunk, store in a vector DB
2. **Retrieve** — Embed the user query, find the top-k most similar chunks
3. **Generate** — Stuff those chunks into a prompt and send to an LLM

It's called "naive" because it doesn't do anything clever about *which* chunks are actually relevant — it just takes the top-k by cosine similarity. This works surprisingly well in many cases, but has specific failure modes we'll expose at the end.

## What you'll learn
- Build a complete RAG pipeline from scratch (step by step)
- Understand every component and what happens if one breaks
- See the failure modes that motivate the more advanced patterns

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import ragkit.config as cfg

# ╔══════════════════════════════════════════╗
# ║  TOGGLE backend here                     ║
# ╚══════════════════════════════════════════╝
cfg.BACKEND = "claude"   # "claude" | "local"

print(f"Backend: {cfg.BACKEND}  |  Device: {cfg.DEVICE}")

## Phase 1 — Indexing

### 1a. Load and chunk the corpus

In [ ]:
from ragkit.data import build_chunked_corpus

texts, metadatas = build_chunked_corpus(chunk_size=200, overlap=40)

print(f"Total chunks: {len(texts)}")
print(f"\nFirst 3 chunks:")
for i, (t, m) in enumerate(zip(texts[:3], metadatas[:3])):
    print(f"\n  [{i}] source={m['source']}  chunk={m['chunk']}")
    print(f"       {t[:120]}...")

### 1b. Embed the chunks

Each chunk gets converted to a 384-dimensional vector by the `all-MiniLM-L6-v2` model running on MPS.

In [ ]:
from ragkit.embeddings import embed
import numpy as np
import time

t0 = time.time()
embeddings = embed(texts)
elapsed = time.time() - t0

print(f"Embedded {len(texts)} chunks in {elapsed:.1f}s")
print(f"Shape: {embeddings.shape}  → {embeddings.shape[0]} chunks × {embeddings.shape[1]} dimensions")
print(f"Memory: {embeddings.nbytes / 1024:.1f} KB")

### 1c. Store in ChromaDB

In [ ]:
from ragkit.vectorstore import build_collection

collection = build_collection(
    name="helios_naive",
    docs=texts,
    metadatas=metadatas,
    persist_dir="../.chroma",
    reset=True,
)
print(f"Collection ready: {collection.count()} chunks indexed")

## Phase 2 — Retrieval

Given a user query, embed it and find the top-k most similar chunks.

In [ ]:
from ragkit.vectorstore import query_collection
from ragkit.pretty import show_hits

query = "What is the fix for the Joint 4 temperature problem?"
hits = query_collection(collection, query, k=5)

show_hits(hits, title=f"Top-5 results for: '{query}'")

In [ ]:
# Let's also inspect the raw similarity scores to understand the distribution
import matplotlib.pyplot as plt

all_hits = query_collection(collection, query, k=collection.count())
scores = [h.score for h in all_hits]

plt.figure(figsize=(8, 3))
plt.hist(scores, bins=20, color='#2b5797', alpha=0.8, edgecolor='white')
plt.axvline(hits[-1].score, color='#e74c3c', ls='--', label=f'Top-5 cutoff: {hits[-1].score:.3f}')
plt.xlabel('Cosine Similarity'); plt.ylabel('Count')
plt.title('Distribution of similarity scores across all chunks')
plt.legend(); plt.tight_layout(); plt.show()

print(f"Top-5 scores: {[round(h.score, 3) for h in hits]}")
print(f"Gap to rank 6: {hits[-1].score - all_hits[5].score:.3f}")

## Phase 3 — Generation

Build the prompt from the retrieved chunks and send to the LLM.

In [ ]:
from ragkit.llm import generate
from ragkit.pretty import show_prompt, show_answer

SYSTEM = """You are a helpful technical support assistant for Helios Robotics.
Answer questions using ONLY the provided context. Be concise and precise.
If the answer is not in the context, say: "I don't have enough information in the provided context."
Always mention relevant part numbers, firmware versions, or incident IDs when available."""

TEMPLATE = """Context from the Helios knowledge base:

{context}

---
Question: {question}"""

context = "\n\n".join(f"[Source: {h.metadata['source']}]\n{h.text}" for h in hits)
prompt = TEMPLATE.format(context=context, question=query)

show_prompt(prompt)

In [ ]:
answer = generate(prompt, system=SYSTEM)
show_answer(answer)

## Full pipeline as a function

Let's package the whole thing cleanly:

In [ ]:
def naive_rag(question: str, collection, k: int = 5, verbose: bool = True) -> str:
    """Complete Naive RAG pipeline: retrieve top-k chunks → generate answer."""
    # Step 1: Retrieve
    hits = query_collection(collection, question, k=k)
    
    # Step 2: Build prompt
    context = "\n\n".join(f"[{h.metadata['source']}]\n{h.text}" for h in hits)
    prompt = TEMPLATE.format(context=context, question=question)
    
    # Step 3: Generate
    answer = generate(prompt, system=SYSTEM)
    
    if verbose:
        show_hits(hits, title=f"Top-{k} hits")
        show_answer(answer)
    
    return answer

# Try several queries
test_queries = [
    "What is the rated payload of the HeliosArm V2?",
    "How long does the HeliosBase M1 battery last?",
]
for q in test_queries:
    print(f"\n{'='*60}\nQ: {q}")
    naive_rag(q, collection, k=3)

## Failure modes of Naive RAG

Now let's deliberately expose the weaknesses. These motivate the more advanced patterns.

In [ ]:
# ── Failure 1: Multi-hop questions ────────────────────────────────────────────
#
# This question requires connecting TWO documents:
#   spec_arm_v2.txt → mentions Sasha Ivanova is working on FW-V2-2.3.2
#   team_engineering.txt → tells us Sasha Ivanova is on the Firmware team
#   project_titan.txt → says the permanent fix is in Project Titan, owned by Lena Bergström
#
# Naive RAG retrieves individual chunks and can't "join" them.

multihop_q = "Who is responsible for the firmware fix for the Joint 4 heat issue, and which team are they on?"
print(f"Multi-hop query: '{multihop_q}'")
print()
naive_rag(multihop_q, collection, k=5)

In [ ]:
# ── Failure 2: Exact part number retrieval ────────────────────────────────────
#
# "HR-REED-UPGRADE" is a specific part number string.
# Semantic similarity may NOT find it if the query is phrased differently.

code_q = "What does part HR-REED-UPGRADE fix and how much does it cost to install?"
print(f"Exact code query: '{code_q}'")
print()
naive_rag(code_q, collection, k=5)

In [ ]:
# ── Failure 3: Lost-in-the-middle ─────────────────────────────────────────────
#
# When k is large, the relevant information may end up buried in the middle
# of the context. LLMs pay less attention to the middle of long contexts.
# Watch the answer quality degrade as k increases.

q = "What is the commissioning procedure step for joint calibration?"
print("Answer with k=3 (focused):")
ans3 = naive_rag(q, collection, k=3, verbose=False)
print(ans3[:300])

print("\nAnswer with k=20 (noisy, relevant chunk may be buried):")
ans20 = naive_rag(q, collection, k=20, verbose=False)
print(ans20[:300])

## Exercise — Why Naive RAG ranks by meaning (RuPaul's Drag Race)

Naive RAG does three things: **embed** the query, score every chunk by **cosine similarity**, and keep the **top-k**. This exercise builds intuition for that ranking — and for the failure that motivates every later notebook — using a tiny hand-built "embedding" space for drag queens.

Each queen is described by a 4-number vector over the dimensions **[comedy, lip-sync, fashion, pageant]**.

1. **Predict first**: For the query *"a hilarious queen who tells jokes and reads people"*, which queen should rank #1? Write your guess as a comment before running.
2. **Implement**: Run the ranking and confirm your prediction against the cosine scores.
3. **Find the failure**: A question like *"which comedy queen also holds a pageant title?"* needs **two** facts that live in **different** chunks. Top-1 retrieval returns only one chunk. Explain in a comment why naive RAG struggles here.
4. **Reflect**: One sentence — why does cosine similarity compare *direction* rather than vector length?

The self-check at the bottom should print ✅ once you run it.

In [ ]:
import numpy as np

def cosine_sim(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# ── A tiny hand-built embedding space ──  dims: [comedy, lip_sync, fashion, pageant]
toy_corpus = {
    "Bianca Del Rio roasts the room with razor-sharp insult comedy": [0.95, 0.20, 0.25, 0.20],
    "Naomi Smalls serves long-legged high-fashion runway looks":     [0.20, 0.35, 0.95, 0.30],
    "Yara Sofia delivers explosive lip syncs and pageant flair":     [0.20, 0.95, 0.40, 0.75],
    "Trinity the Tuck is a polished pageant queen with clean looks":  [0.30, 0.35, 0.70, 0.95],
}

# ── Task 1: predict the #1 result (write before running) ──────────────────────
# My prediction for "a hilarious queen who tells jokes": ____________________

# ── Task 2: build a query vector and rank by cosine similarity ────────────────
query = "a hilarious queen who tells jokes and reads people"
query_vec = [0.95, 0.10, 0.15, 0.10]   # ← emphasises the 'comedy' dimension

print(f"Query: {query!r}")
print("-" * 60)
ranked = sorted(((cosine_sim(query_vec, v), t) for t, v in toy_corpus.items()), reverse=True)
for sim, text in ranked:
    bar = "█" * int(sim * 20)
    print(f"  {sim:.3f} {bar:<20} {text}")

# ── Task 3: the multi-hop failure (comment) ───────────────────────────────────
# "Which comedy queen ALSO holds a pageant title?" needs the comedy fact AND the
# pageant fact, but top-1 returns a single chunk. Why does naive RAG miss it?
#   Your answer:

# ── Task 4: direction vs magnitude (comment) ──────────────────────────────────
#   Your answer:

# ── Self-check ────────────────────────────────────────────────────────────────
assert "Bianca" in ranked[0][1], "the comedy query should rank the insult-comic first"
assert ranked[0][0] > ranked[1][0], "top-1 should clearly beat the runner-up"
print("\n✅ Exercise checks passed!")

## Tradeoffs

| Aspect | Naive RAG |
|---|---|
| **Setup complexity** | ★☆☆☆☆ Very simple |
| **Retrieval quality** | ★★★☆☆ Good for simple factual questions |
| **Multi-hop reasoning** | ★☆☆☆☆ Poor — can't connect documents |
| **Exact string matching** | ★★☆☆☆ Inconsistent — semantic only |
| **Latency** | ★★★★★ Fast — one embedding + one LLM call |
| **Cost** | ★★★★★ Cheap — minimal tokens |
| **When to use** | Single-document Q&A, simple FAQs, quick prototypes |

## Exercises

1. **Change `k`**: Try `k=1`, `k=3`, `k=10` on the same query. How does answer quality change?
2. **Change `chunk_size`**: Rebuild the collection with `chunk_size=50` and `chunk_size=500`. How does retrieval quality change?
3. **Adversarial query**: Ask something not in the corpus (e.g. "What is the price of the HeliosArm V2?"). Does the model correctly say it doesn't know, or does it hallucinate?
4. **Backend comparison**: Run the same query with `cfg.BACKEND = "claude"` and `cfg.BACKEND = "local"`. Compare answer quality and latency.

**Next:** [02_retrieve_and_rerank.ipynb](02_retrieve_and_rerank.ipynb) — fix the ranking quality issue with a cross-encoder reranker.